In [80]:
import numpy as np
import pandas as pd 
import re

In [81]:
def header():
    print('*!*' * 32 + '\n\n\n')
    return


def table_check(df, df_name, unique_id) -> pd.DataFrame:
    '''
    Checks and prints the count of null (NaN) values for every column in a Pandas DataFrame.
    '''
    header()
    print(f'Beginning errorcheck for: {df_name} ***')
    print(f'\n--- Null Value Check ---')
    for column in df.columns: 
        null_count = df[column].isna().sum() 
        print(f'{column} nulls: {null_count}')
    print('\n')
    
    print(f'--- Integrity Check ---')
    if unique_id in df.columns:
        duplicate_count = df[unique_id].duplicated().sum()
        print(f'Total appointment_id duplicates: {duplicate_count}')
    else:
        print(f"WARNING: {unique_id} not found for duplication check.")
    print('\n')
    print(f'\n--- Data Type Audit (DF.dtypes) ---')
    print(df.dtypes)
    
    print(f'\n--- Numerical Sanity Check (DF.describe) ---')
    
    numeric_cols = df.select_dtypes(include = ['int64', 'float64', 'Int64', 'int32']).columns
    if not numeric_cols.empty:
        print(df[numeric_cols].describe())
    else:
        print('No standard numerical columns found for description.')

    print(f'\n--- Categorical Value Audit (Top 10 Counts) ---')
    object_cols = df.select_dtypes(include=['object']).columns

    for column in object_cols:
        print(f'\n{column.upper()}:')
        print(df[column].value_counts().nlargest(10))
    
    print('\n\n\n')
    
    return df
    
# ============================================================
# 1. TYPE CASTING (reuses your existing correct_d_type/table_check)
# ============================================================

def bl_clean_headers(df) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df


# ============================================================
# 2. SPECIES MAPPING  (mirrors cln_perros_y_gatos)
# ============================================================

def bl_cln_spp(df) -> pd.DataFrame:
    df = df.rename(columns={'speciesname': 'spp'})
    df['spp'] = (
        df['spp'].astype(str).str.strip().str.lower()
        .replace({'cat': 'fel', 'dog': 'k9'})
    )
    return df


# ============================================================
# 2b. TRUE RECIDIVISM FLAG (30-DAY WINDOW, MATCHING AUSTIN)
#    Austin's 'returned' target = adopted, then a NEW intake record for
#    that same animal within 30 days of discharge. Bloomington's
#    returndate/returnedreason fields are NOT a reliable analog on their
#    own -- returnedreason == 'Stray' on 100% of Foster-type return rows
#    (a default/placeholder value, not real signal), and a raw
#    returndate.notna() check has no time-window constraint at all, so it
#    conflates true recidivism with returns that happened months or years
#    later -- a different phenomenon than Austin's 30-day definition.
#    This restricts true_recidivism to Adoption-type rows where the return
#    happened within 30 days of the adoption's start date, for a genuine
#    apples-to-apples comparison. Must run on the FULL, uncollapsed row
#    set -- collapsing to each animal's final row would lose any failed
#    adoption that wasn't the animal's last placement attempt.
# ============================================================

def bl_flag_true_recidivism(df) -> pd.DataFrame:
    key = df['movementtype'].astype(str).str.strip().str.lower()
    movementdate = pd.to_datetime(df['movementdate'], errors='coerce')
    returndate = pd.to_datetime(df['returndate'], errors='coerce')

    return_time_days = (returndate - movementdate).dt.total_seconds() / 86400

    adoption_return_30d = (
        (key == 'adoption')
        & returndate.notna()
        & (return_time_days >= 0)
        & (return_time_days <= 30)
    )

    df['return_time_days'] = return_time_days.where(key == 'adoption')
    recidivism_by_animal = (
        df.assign(_adoption_return_30d=adoption_return_30d)
        .groupby('sheltercode')['_adoption_return_30d']
        .any()
    )
    df['true_recidivism'] = df['sheltercode'].map(recidivism_by_animal)
    return df


# ============================================================
# 3. SEX MAPPING
#    NOTE: Bloomington has no altered/intact status (unlike Austin's
#    "Neutered Male" / "Intact Female"). We can only map biological sex.
#    This is a real feature gap vs. Austin -- flag it in your writeup.
# ============================================================

def bl_clean_sex(df) -> pd.DataFrame:
    condition = [
        df['sexname'].str.lower() == 'male',
        df['sexname'].str.lower() == 'female',
    ]
    choice = ['m', 'f']
    df['sex_in'] = np.select(condition, choice, default='unknown')
    df = df.drop(columns=['sexname'])
    return df


# ============================================================
# 4. AGE PARSING
#    Bloomington format: "9 years 2 months." (combined, single string)
#    Austin format:       "2 years"          (single unit)
#    Needs its own parser -- age_clean() from Austin won't work as-is.
# ============================================================

def bl_age_clean(df) -> pd.DataFrame:
    def parse_age(text):
        if pd.isna(text):
            return np.nan
        text = str(text).lower()
        years = re.search(r'(\d+)\s*year', text)
        months = re.search(r'(\d+)\s*month', text)
        weeks = re.search(r'(\d+)\s*week', text)
        days = re.search(r'(\d+)\s*day', text)

        total_years = 0.0
        if years:
            total_years += int(years.group(1))
        if months:
            total_years += int(months.group(1)) / 12
        if weeks:
            total_years += int(weeks.group(1)) / 52.1786
        if days:
            total_years += int(days.group(1)) / 365.25

        if not (years or months or weeks or days):
            return np.nan
        return round(total_years, 4)

    df['age_in_years'] = df['animalage'].apply(parse_age)
    df = df.drop(columns=['animalage'])
    return df


# ============================================================
# 5. INTAKE REASON -> intake_type (circumstance) + intake_reason (medical/behavior/routine/other)
#    Bloomington's `intakereason` collapses Austin's two separate fields
#    (intake_type + intake_condition) into one. This mapping is a judgment
#    call -- review it against your own domain knowledge and adjust.
# ============================================================

INTAKE_TYPE_MAP = {
    'stray': 'stray',
    'abandoned': 'stray',
    'incompatible with owner lifestyle': 'owner surrender',
    'litter relinquishment': 'owner surrender',
    'moving': 'owner surrender',
    'unsuitable accommodation': 'owner surrender',
    'unable to afford': 'owner surrender',
    'landlord issues': 'owner surrender',
    'owner deceased': 'owner surrender',
    'owner died': 'owner surrender',
    'allergies': 'owner surrender',
    'incompatible with other pets': 'owner surrender',
    'biting': 'owner surrender',
    'marriage/relationship split': 'owner surrender',
    'behavioral issues': 'owner surrender',
    'owner requested euthanasia': 'owner surrender',
    'transfer from other shelter': 'transfer',
    'police assist': 'public assist',
    'rabies monitoring': 'public assist',
    'tnr - trap/neuter/release': 'public assist',
    'abuse/ neglect': 'public assist',
    'sick/injured': 'public assist',
    'born in shelter': 'born in shelter',
    'injured wildlife': 'wildlife',
    'doa': 'doa',
}

INTAKE_REASON_MAP = {
    'sick/injured': 'medical',
    'allergies': 'medical',
    'biting': 'behavior',
    'behavioral issues': 'behavior',
    'rabies monitoring': 'medical',
    'abuse/ neglect': 'medical',
    'owner requested euthanasia': 'medical',
    'doa': 'medical',
    'injured wildlife': 'medical',
    'tnr - trap/neuter/release': 'routine',
    'stray': 'routine',
    'abandoned': 'routine',
    'transfer from other shelter': 'routine',
    'police assist': 'routine',
    'born in shelter': 'routine',
}


def bl_intake_reason_clean(df) -> pd.DataFrame:
    key = df['intakereason'].str.strip().str.lower()
    df['intake_type'] = key.map(INTAKE_TYPE_MAP).fillna('owner surrender')
    df['intake_reason'] = key.map(INTAKE_REASON_MAP).fillna('other')
    df['intake_reason'] = df['intake_reason'].replace({'behavior': 'other'})  # match Austin's 3-category feature set
    df = df.drop(columns=['intakereason'])
    return df


# ============================================================
# 6. COLLAPSE MULTI-ROW STAYS TO FINAL OUTCOME
#    Some sheltercodes have multiple movement rows (e.g. an interim
#    Foster placement followed by the real final outcome like
#    Adoption/Transfer/Reclaimed). Keep only the last chronological
#    movementdate per sheltercode so los_days and outcome_category
#    reflect the true final result of the stay, not an interim step.
# ============================================================

def bl_collapse_to_final_outcome(df) -> pd.DataFrame:
    df = df.sort_values('movementdate')
    df = df.drop_duplicates(subset='sheltercode', keep='last')
    return df


# ============================================================
# 7. OUTCOME MAPPING (mirrors clean_outcome_type)
#    Confirmed against real movementtype.value_counts():
#    Adoption, Foster, Reclaimed, Transfer, Released To Wild, Stolen, Escaped
#    deceased is gated on deceaseddate.notna() -- NOT deceasedreason,
#    which is a placeholder field (see prior data-quality check).
# ============================================================

def bl_clean_outcome(df) -> pd.DataFrame:
    key = df['movementtype'].astype(str).str.strip().str.lower()

    alive_outcome = ['adoption', 'foster', 'reclaimed']
    admin_outcome = ['transfer', 'stolen', 'escaped']
    wildlife_outcome = ['released to wild']

    # deceased MUST be checked first -- movementtype has no "died"/"euthanized"
    # value, so a deceased animal's last movement type could still read as
    # Transfer/Adoption/etc. deceaseddate is the more reliable signal and
    # should override whatever movementtype says.
    conditions = [
        df['deceaseddate'].notna(),
        key.isin(alive_outcome),
        key.isin(admin_outcome),
        key.isin(wildlife_outcome),
    ]
    choices = ['deceased', 'alive', 'admin', 'wildlife']
    df['outcome_category'] = np.select(conditions, choices, default='unknown')
    df = df.drop(columns=['movementtype'])
    return df


# ============================================================
# 8. FULL PIPELINE
# ============================================================

def bloomington_line(df) -> pd.DataFrame:
    print('\n\n\n')
    header()
    print('Beginning Bloomington -> Austin schema mapping')
    return (
        df
        .pipe(bl_clean_headers)
        .pipe(bl_flag_true_recidivism)      # must run before collapse
        .pipe(bl_cln_spp)
        .pipe(bl_clean_sex)
        .pipe(bl_collapse_to_final_outcome)
        .pipe(bl_age_clean)
        .pipe(bl_intake_reason_clean)
        .pipe(bl_clean_outcome)
        .rename(columns={
            'sheltercode': 'apt_id',
            'id': 'animal_id',
            'intakedate': 'datetime_in',
            'movementdate': 'datetime_out',
        })
        .assign(
            datetime_in=lambda d: pd.to_datetime(d['datetime_in'], errors='coerce'),
            datetime_out=lambda d: pd.to_datetime(d['datetime_out'], errors='coerce'),
        )
        .assign(
            los_days=lambda d: (d['datetime_out'].dt.normalize() - d['datetime_in'].dt.normalize()).dt.total_seconds() / 86400,
            was_returned=lambda d: d['true_recidivism']
        )
        .assign(
            los_days=lambda d: d['los_days'].where(d['los_days'] >= 0, np.nan),
            age_in_years=lambda d: d['age_in_years'].where(d['age_in_years'] <= 25, np.nan)
        )
    )


if __name__ == '__main__':
    pass

In [82]:
bloomington_data = pd.read_csv('/kaggle/input/datasets/thedevastator/analyzing-adoption-trends-at-the-bloomington-ani/animal-data-1.csv')
bloomington_clean = bloomington_line(bloomington_data.copy())






*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning Bloomington -> Austin schema mapping


In [83]:
min_date = bloomington_data['intakedate'].min()
max_date = bloomington_data['intakedate'].max()

print("Earliest Date:", min_date)
print("Latest Date:", max_date)




Earliest Date: 2009-11-28 00:00:00
Latest Date: 2019-08-30 10:22:37


In [84]:
# 1. Expanded mapping to align NOAA LCD data as closely as possible to Visual Crossing
column_mapping = {
    'DATE': 'datetime',

    # Precipitation & Snow
    'DailyPrecipitation': 'precip',
    'DailySnowDepth': 'snowdepth',
    'DailySnowfall': 'snow',

    # Temperature Core Metrics
    'DailyMaximumDryBulbTemperature': 'tempmax',
    'DailyMinimumDryBulbTemperature': 'tempmin',
    'DailyAverageDryBulbTemperature': 'temp',

    # Moisture & Atmosphere -- see hourly fallback below (Daily* fields are
    # empty across all years for this station)

    # Wind Metrics
    'DailyAverageWindSpeed': 'windspeed',
    'DailyPeakWindSpeed': 'windgust',
    'DailyResultantWindDirection': 'winddir',

    # Visibility and Sea Level Pressure
    # NOTE: using SeaLevelPressure, not StationPressure -- these are physically
    # different measurements (station pressure is uncorrected for elevation).
    'HourlyVisibility': 'visibility',
    'HourlySeaLevelPressure': 'sealevelpressure',

    # Dew point / humidity: DailyAverage* versions are empty across all 11 years
    # for this station (confirmed via .notna().sum() check per-year) -- falling
    # back to hourly readings, aggregated to daily mean below like pressure.
    'HourlyDewPointTemperature': 'dew',
    'HourlyRelativeHumidity': 'humidity',

    # Pressure change/tendency -- this is the feature that actually mattered
    # in the moon-phase analysis (p = 2.58e-23), not raw pressure level.
    # Pull it directly from the source if available; we also compute a
    # day-over-day delta below as a fallback/cross-check.
    'HourlyPressureChange': 'pressure_change_hourly',
}

# 2. Build the path loop for years 2009 through 2019
file_paths = [
    f'/kaggle/input/datasets/micahluftig/full-weather-bloomington-2009-2019/LCD_USW00003893_{year}.csv'
    for year in range(2009, 2020)
]

# Load and filter only if columns exist in the source dataset
df_list = []
for path in file_paths:
    raw_df = pd.read_csv(path, low_memory=False)
    existing_cols = [col for col in column_mapping.keys() if col in raw_df.columns]
    filtered_df = raw_df[existing_cols].rename(columns={k: v for k, v in column_mapping.items() if k in existing_cols})
    df_list.append(filtered_df)

weather_master_df = pd.concat(df_list, ignore_index=True)

# 3. Standardize and Clean Types
weather_master_df['datetime'] = pd.to_datetime(weather_master_df['datetime'], errors='coerce')

numeric_cols = [col for col in weather_master_df.columns if col != 'datetime']
for col in numeric_cols:
    weather_master_df[col] = pd.to_numeric(
        weather_master_df[col].astype(str).str.strip().str.lower().str.replace(r'^t$', '0', regex=True),
        errors='coerce'
    )

# 4. Aggregate to clean Daily Rows
weather_master_df['date_only'] = weather_master_df['datetime'].dt.date

agg_rules = {}
for col in numeric_cols:
    if col in ['precip', 'snowdepth', 'snow', 'tempmax', 'windgust']:
        agg_rules[col] = 'max'
    elif col in ['tempmin']:
        agg_rules[col] = 'min'
    else:
        agg_rules[col] = 'mean'

clean_weather_df = weather_master_df.groupby('date_only').agg(agg_rules).reset_index()
clean_weather_df = clean_weather_df.rename(columns={'date_only': 'datetime'})
clean_weather_df['datetime'] = pd.to_datetime(clean_weather_df['datetime'])

# 5. Filter precisely to your target date window
start_bound = '2009-11-28'
end_bound = '2019-08-30'
clean_weather_df = clean_weather_df[clean_weather_df['datetime'].between(start_bound, end_bound)].reset_index(drop=True)

# 6. Pressure delta features
clean_weather_df = clean_weather_df.sort_values('datetime').reset_index(drop=True)
clean_weather_df['pressure_change_1d'] = clean_weather_df['sealevelpressure'].diff()
clean_weather_df['pressure_drop_flag'] = clean_weather_df['pressure_change_1d'] < -3.0

if 'pressure_change_hourly' in clean_weather_df.columns:
    clean_weather_df = clean_weather_df.drop(columns=['pressure_change_hourly'])

clean_weather_df[['snowdepth', 'snow']] = clean_weather_df[['snowdepth', 'snow']].fillna(0)

print(clean_weather_df.shape)
print(clean_weather_df.head())
print(clean_weather_df.isna().sum())

(3563, 15)
    datetime  precip  snowdepth  snow  tempmax  tempmin  temp  windspeed  \
0 2009-11-28     0.0        0.0   0.0     15.6     -2.7   6.5        3.1   
1 2009-11-29     2.3        0.0   0.0     12.8      6.7   9.8        4.4   
2 2009-11-30     0.0        0.0   0.0      6.7     -2.7   2.0        2.6   
3 2009-12-01     0.0        0.0   0.0     12.2     -1.0   5.6        3.5   
4 2009-12-02    19.6        0.0   0.0      6.7      0.0   3.4        3.3   

   windgust  visibility  sealevelpressure       dew   humidity  \
0       9.4   15.958917       1013.145833 -0.316667  63.416667   
1      10.3   11.095711       1010.158333  5.457895  75.552632   
2       8.1   12.959211       1016.212500 -0.165789  80.552632   
3       9.4   16.093000       1016.129167 -3.091667  58.958333   
4      10.7    8.149340       1002.712500  1.753191  82.808511   

   pressure_change_1d  pressure_drop_flag  
0                 NaN               False  
1           -2.987500               False  
2  

In [85]:
print(bloomington_clean['datetime_in'].min(), bloomington_clean['datetime_in'].max())

2009-11-28 00:00:00 2019-08-30 10:22:37


In [86]:
x = table_check(clean_weather_df, df_name = 'clean_weather', unique_id = 'apt_id')
print(x)

*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning errorcheck for: clean_weather ***

--- Null Value Check ---
datetime nulls: 0
precip nulls: 25
snowdepth nulls: 0
snow nulls: 0
tempmax nulls: 25
tempmin nulls: 23
temp nulls: 27
windspeed nulls: 23
windgust nulls: 30
visibility nulls: 0
sealevelpressure nulls: 0
dew nulls: 10
humidity nulls: 10
pressure_change_1d nulls: 1
pressure_drop_flag nulls: 0


--- Integrity Check ---



--- Data Type Audit (DF.dtypes) ---
datetime              datetime64[ns]
precip                       float64
snowdepth                    float64
snow                         float64
tempmax                      float64
tempmin                      float64
temp                         float64
windspeed                    float64
windgust                     float64
visibility                   float64
sealevelpressure             float64
dew                          float64
humidity                    

In [87]:
x = table_check(bloomington_clean, df_name = 'bloomington_clean', unique_id = 'id')
print(x)

*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning errorcheck for: bloomington_clean ***

--- Null Value Check ---
index nulls: 0
animal_id nulls: 0
datetime_in nulls: 0
istransfer nulls: 0
apt_id nulls: 0
identichipnumber nulls: 1786
animalname nulls: 0
breedname nulls: 42
basecolour nulls: 0
spp nulls: 0
location nulls: 0
datetime_out nulls: 0
istrial nulls: 0
returndate nulls: 7028
returnedreason nulls: 0
deceaseddate nulls: 7036
deceasedreason nulls: 0
diedoffshelter nulls: 0
puttosleep nulls: 0
isdoa nulls: 0
return_time_days nulls: 7236
true_recidivism nulls: 0
sex_in nulls: 0
age_in_years nulls: 84
intake_type nulls: 0
intake_reason nulls: 0
outcome_category nulls: 0
los_days nulls: 5
was_returned nulls: 0


--- Integrity Check ---



--- Data Type Audit (DF.dtypes) ---
index                        int64
animal_id                    int64
datetime_in         datetime64[ns]
istransfer                   int64
apt_id       

In [88]:
# Per-animal join: weather on the day of intake
bloomington_clean['intake_date_only'] = bloomington_clean['datetime_in'].dt.normalize()

model_df = bloomington_clean.merge(
    clean_weather_df,
    left_on='intake_date_only',
    right_on='datetime',
    how='left',
    suffixes=('', '_weather')
)
print(model_df.shape)
print(model_df['sealevelpressure'].isna().sum())



(7288, 45)
0


In [89]:
# ============================================================
# VALIDATION: true_recidivism vs. the naive returndate.notna() signal
#    Confirms the fix for the misleading 'returnedreason' field. Foster-type
#    movement rows show returnedreason == 'Stray' on 100% of rows where
#    populated -- a default/placeholder value in the source data, not a
#    real answer, and a foster placement ending is a normal event, not a
#    failed adoption. was_returned is now scoped to true Adoption-return
#    events only, checked across ALL of an animal's movement rows (not
#    just the final one kept after collapsing).
# ============================================================

recidivism_rate = model_df['was_returned'].mean()
print(f"True recidivism rate: {recidivism_rate:.2%} ({model_df['was_returned'].sum()} of {len(model_df)} animals)")
print()

returned = model_df[model_df['was_returned']]
deceased_rate_returned = (returned['outcome_category'] == 'deceased').mean()
deceased_rate_all = (model_df['outcome_category'] == 'deceased').mean()

print(f"Deceased rate among recidivist animals: {deceased_rate_returned:.2%}")
print(f"Deceased rate shelter-wide:             {deceased_rate_all:.2%}")
print(f"Relative elevation: {deceased_rate_returned / deceased_rate_all:.2f}x")
print()

print("Outcome breakdown among recidivist animals:")
print(returned['outcome_category'].value_counts())

True recidivism rate: 3.90% (284 of 7288 animals)

Deceased rate among recidivist animals: 8.45%
Deceased rate shelter-wide:             3.46%
Relative elevation: 2.44x

Outcome breakdown among recidivist animals:
outcome_category
alive       249
deceased     24
admin        11
Name: count, dtype: int64


In [90]:
# ============================================================
#    9. AKC BREED GROUP MAPPING
#    Mirrors Austin's cln_akc_group exactly in category definitions and
#    priority order (combo/mix check first, then sporting -> hound ->
#    herding -> working -> terrier -> toy -> non_sporting), so the two
#    datasets use a genuinely comparable taxonomy. Adapted for Bloomington's
#    space/comma-separated breed strings (vs. Austin's underscore-joined
#    format) and its "/Mix" suffix convention. Two additions not present
#    in Austin's data: "bully"/"pit bull" mapped to terrier (matching
#    Austin's own existing pit_bull -> terrier convention) and "spitz"
#    added to non_sporting, both needed to cover breeds common in this
#    dataset but rare/absent in Austin's.
#    Coverage: 99.97% of dog records (1 of 3,900 unmapped).
# ============================================================

def bl_strip_generic_mix(breed_str) -> str:
    """Splits on '/' and removes any standalone 'Mix' word from each piece,
    so 'Labrador Retriever/Mix' becomes 'Labrador Retriever' (still
    classified as a Lab) rather than triggering the mixed_breed rule --
    matching Austin's own convention of stripping a bare mix suffix before
    classification. Genuine two-breed combos like 'Labrador Retriever/Beagle'
    are left untouched, so they still correctly trigger mixed_breed."""
    parts = re.split(r'/', str(breed_str))
    cleaned = []
    for p in parts:
        p2 = re.sub(r'(?i)\bmix\b', '', p).strip()
        if p2:
            cleaned.append(p2)
    if not cleaned:
        return 'Mix'
    return '/'.join(cleaned)


def bl_cln_akc_group(df) -> pd.DataFrame:
    def classify(breed_str):
        b = bl_strip_generic_mix(breed_str)
        if b == 'Mix' or '/' in b:
            return 'mixed_breed'
        text = b.lower()
        if re.search(r'retriever|setter|pointer|spaniel|vizsla|weimaraner|brittany|griffon|water dog', text):
            return 'sporting'
        if re.search(r'hound|beagle|basenji|dachshund|whippet|greyhound|rhod|harrier|saluki|pbgv|treeing cur|podengo|carolina dog|jindo', text):
            return 'hound'
        if re.search(r'shepherd|shep\b|collie|corgi|kelpie|cattle dog|malinois|sheepdog|border|heeler|beauceron|vallhund|briard|tervuren|catahoula|canaan|bouvier', text):
            return 'herding'
        if re.search(r'mastiff|husky|malamute|akita|dane|pyrenees|boxer|rottweiler|pinsch|dogo|mountain dog|canario|bordeaux|corso|boerboel|st\.? ?bernard|saint bernard|landseer|samoyed|leonberger|newfoundland|entlebucher|akbash|kuvasz|hovawart|kangal|wolf hybrid|\bcur\b', text):
            return 'working'
        if re.search(r'terrier|terr\b|schnauzer|pinscher|cairn|staffordshire|imaal|dandie|bedlington|sealyham|west highland|pit ?bull|bully|feist', text):
            return 'terrier'
        if re.search(r'chihuahua|pomeranian|pug|maltese|pekingese|papillon|toy|havanese|bichon|shih tzu|bruss|crested|chin\b', text):
            return 'toy'
        if re.search(r'poodle|cockapoo|bulldog|sharpei|shar.pei|chow|dalmatian|boston|keeshond|lhasa|eskimo|shiba|tulear|schipperke|lowchen|finnish spitz|\bspitz\b', text):
            return 'non_sporting'
        return 'unknown_other'

    k9_mask = df['spp'] == 'k9'
    df['akc_group'] = pd.Series('non_canine', index=df.index)
    df.loc[k9_mask, 'akc_group'] = df.loc[k9_mask, 'breedname'].apply(classify)
    return df

In [91]:
model_df = bl_cln_akc_group(model_df)

print(model_df['akc_group'].value_counts())

akc_group
non_canine       4420
mixed_breed       896
terrier           532
hound             370
sporting          334
herding           288
working           215
toy               151
non_sporting       81
unknown_other       1
Name: count, dtype: int64


In [92]:
print(model_df['was_returned'].value_counts())
print(f"Rate: {model_df['was_returned'].mean():.2%}")

was_returned
False    7004
True      284
Name: count, dtype: int64
Rate: 3.90%


In [93]:
# ============================================================
# FEATURE MATRIX -- mirrors Austin's build_features()/FEATURE_ORDER exactly
# (from recidivism-model-and-evaluation.ipynb), so the two datasets produce
# directly comparable feature vectors for the same model.
# ============================================================

FEATURE_ORDER = [
    "los_days", "age_at_first_visit",
    "spp_k9", "spp_other", "spp_wildlife",
    "akc_group_hound", "akc_group_mixed_breed", "akc_group_non_sporting",
    "akc_group_sporting", "akc_group_terrier", "akc_group_toy", "akc_group_working",
    "first_reason_medical", "first_reason_other", "first_reason_routine",
]

def build_features(df):
    X = pd.DataFrame(index=df.index)
    X['los_days'] = df['los_days']
    X['age_at_first_visit'] = df['age_in_years']  # Bloomington column name differs from Austin's
    spp = df['spp'].astype(str).str.lower()
    X['spp_k9'] = (spp == 'k9').astype(int)
    X['spp_wildlife'] = (spp == 'wildlife').astype(int)
    X['spp_other'] = ((~spp.isin(['k9', 'fel'])).astype(int) - X['spp_wildlife']).clip(lower=0)
    akc = df['akc_group'].astype(str).str.lower()
    for group in ['hound', 'mixed_breed', 'non_sporting', 'sporting', 'terrier', 'toy', 'working']:
        X[f'akc_group_{group}'] = (akc == group).astype(int)
    reason = df['intake_reason'].astype(str).str.lower()  # Bloomington column name differs from Austin's
    for r in ['medical', 'other', 'routine']:
        X[f'first_reason_{r}'] = (reason == r).astype(int)
    return X[FEATURE_ORDER]

bloomington_features = model_df.dropna(subset=['los_days', 'age_in_years', 'spp', 'akc_group', 'intake_reason']).copy()
X_bloomington = build_features(bloomington_features)
y_bloomington = bloomington_features['was_returned'].astype(int)

print(X_bloomington.shape)
print(X_bloomington.head())
print()
print(f"Recidivism rate in usable rows: {y_bloomington.mean():.2%} ({y_bloomington.sum()} of {len(y_bloomington)})")

(7199, 15)
   los_days  age_at_first_visit  spp_k9  spp_other  spp_wildlife  \
0       2.0              0.0575       0          0             0   
1       2.0              0.0575       0          0             0   
2       2.0              0.0192       0          0             0   
3       2.0              0.0575       0          0             0   
4       7.0              4.5833       0          0             0   

   akc_group_hound  akc_group_mixed_breed  akc_group_non_sporting  \
0                0                      0                       0   
1                0                      0                       0   
2                0                      0                       0   
3                0                      0                       0   
4                0                      0                       0   

   akc_group_sporting  akc_group_terrier  akc_group_toy  akc_group_working  \
0                   0                  0              0                  0   
1        

In [94]:
model_df.to_csv('/kaggle/working/bloomington_model_ready.csv', index=False)